# Civic Reranker Map Inspector

Sanity-check tool for `civic` (haversine) vs `civic_osrm` (real travel distance)
side by side on an actual map, for one user at a time -- the visual equivalent
of https://andreafooo.github.io/Thesis_Balancing_Unfairness_POI/poi_visualization_demo_tool.html
adapted to this project's datasets and OSRM-powered civic reranker.

Set `DATASET` / `MODEL_DIRECTORY` / `USER_ID` in the last cell and a map
renders with one togglable layer per method:
- **Baseline** (grey, dashed) -- straight-line order, no geo re-ranking.
- **Civic (haversine)** (orange, dashed) -- `civic_reranker.py`'s greedy
  nearest-neighbour order using as-the-crow-flies distance.
- **Civic (OSRM)** (blue, solid) -- same greedy logic, but distances (and the
  drawn route) come from OSRM's real street/path network.

The Civic (OSRM) route line is the *actual routed path* (via OSRM `/route`,
`overview=full`), not a straight line -- so if it detours around a river,
follows a road grid, or refuses to jump a highway that the haversine version
cuts straight through, that's the tool doing its job. A big gap between the
`sequential_dist_haversine_km` and `sequential_dist_osrm_km` numbers in the
stats table below the map is a flag worth looking at (e.g. an unroutable
patch of the network falling back to haversine, or a real detour haversine
can't see).

Requires the dataset's OSRM container to be running (see `osrm/docker-compose.yml`)
for the OSRM route/line and `sequential_dist_osrm_km` to reflect real
distances -- otherwise this degrades gracefully to the haversine fallback
`osrm_client.py` already implements everywhere else.

In [ ]:
import os
import json

import folium
import pandas as pd
import requests
from IPython.display import display

from globals import (
    BASE_DIR,
    recommendation_dirpart,
    available_datasets,
    top_k_eval,
    OSRM_HOST,
    OSRM_PORTS,
    OSRM_DEFAULT_PROFILE,
)
from postprocess_baseline_top_k import dataset_metadata
from civic_reranker import load_coordinates
from evaluation_metrics import haversine
from osrm_client import OSRMClient

# color/line-style per method -- civic_osrm is drawn solid since its line is
# the real routed path; the other two are dashed straight-line orderings.
METHOD_STYLES = {
    "baseline":   {"color": "#7f7f7f", "dash": "4,8", "label": "Baseline"},
    "civic":      {"color": "#e07b39", "dash": "4,4", "label": "Civic (haversine)"},
    "civic_osrm": {"color": "#1f77b4", "dash": None,  "label": "Civic (OSRM)"},
}
DEFAULT_METHODS = ("baseline", "civic", "civic_osrm")

## Loading helpers

In [ ]:
def recommender_dirs(dataset):
    """{model_name: directory} for one dataset, from directory names on disk."""
    data = dataset_metadata(dataset, recommendation_dirpart)
    return {d["model"]: d["directory"] for d in data}


def load_top_k(dataset, model_directory, method, k=top_k_eval):
    """{user_id: [item_id, ...]} (rank order, truncated to the top `k`) for
    one method's saved output, or {} if that method wasn't run for this
    model (e.g. civic_osrm not yet computed for every model). Saved files
    hold up to top_k_resample candidates per user, but every other
    evaluation in this project (see offline_evaluation.ipynb's
    top_k_to_df) scores only the top top_k_eval -- default here matches
    that so the map reflects what actually gets evaluated/served."""
    path = os.path.join(
        BASE_DIR, f"{dataset}_dataset", recommendation_dirpart,
        model_directory, method, "top_k_recommendations.json",
    )
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        data = json.load(f)
    result = {}
    for user, items in data.items():
        if items and isinstance(items[0], list):
            items = items[0]  # some stages nest as [[...]]
        result[user] = items[:k] if k is not None else items
    return result


def user_ids_for(dataset, model_directory, method="baseline"):
    return sorted(load_top_k(dataset, model_directory, method).keys(), key=lambda u: (len(u), u))

## OSRM helpers -- real route geometry + sequential travel distance

In [ ]:
def osrm_route_geometry(dataset, profile, coords_sequence):
    """
    Real street/path polyline (list of (lat, lon)) following coords_sequence
    leg by leg via OSRM's /route (overview=full). Falls back to a straight
    segment for any leg OSRM can't route (unreachable, container down, etc.)
    so the map always draws something -- a leg that falls back is exactly
    the kind of thing this tool exists to surface.
    """
    if dataset not in OSRM_PORTS or profile not in OSRM_PORTS[dataset]:
        return list(coords_sequence)
    base_url = f"http://{OSRM_HOST}:{OSRM_PORTS[dataset][profile]}"
    line = [coords_sequence[0]]
    for (lat1, lon1), (lat2, lon2) in zip(coords_sequence[:-1], coords_sequence[1:]):
        url = f"{base_url}/route/v1/{profile}/{lon1},{lat1};{lon2},{lat2}?overview=full&geometries=geojson"
        try:
            resp = requests.get(url, timeout=5)
            resp.raise_for_status()
            data = resp.json()
            if data.get("code") == "Ok":
                leg_points = [(lat, lon) for lon, lat in data["routes"][0]["geometry"]["coordinates"]]
                line.extend(leg_points[1:])
                continue
        except requests.exceptions.RequestException:
            pass
        line.append((lat2, lon2))  # fallback: straight segment for this leg
    return line


def sequential_distances(item_ids, item_coords, osrm_client=None):
    """(haversine_total_km, osrm_total_km_or_None) walking item_ids top to
    bottom in order -- what the user would actually cover visiting the list
    as ranked. Reuses osrm_client's on-disk pair cache (keyed by real item
    ids), so this is fast for any pair a prior batch run already resolved."""
    ids_with_coords = [i for i in item_ids if i in item_coords]
    if len(ids_with_coords) < 2:
        return 0.0, None
    pairs = list(zip(ids_with_coords[:-1], ids_with_coords[1:]))
    hav_total = sum(haversine(*item_coords[a], *item_coords[b]) for a, b in pairs)
    osrm_total = None
    if osrm_client is not None:
        osrm_total = sum(
            osrm_client.distance_km(a, item_coords[a], b, item_coords[b], haversine)
            for a, b in pairs
        )
    return hav_total, osrm_total

## Map builder

In [ ]:
def build_user_map(dataset, model_directory, user_id, methods=DEFAULT_METHODS, osrm_profile=OSRM_DEFAULT_PROFILE, k=top_k_eval):
    coords_df = load_coordinates(dataset)
    item_coords = dict(zip(
        coords_df["item_id:token"], zip(coords_df["lat:float"], coords_df["lon:float"]),
    ))
    osrm_client = OSRMClient(dataset, profile=osrm_profile)

    fmap = None
    stats_rows = []

    for method in methods:
        top_k = load_top_k(dataset, model_directory, method, k=k)
        item_ids = [i for i in top_k.get(user_id, []) if i in item_coords]
        if not item_ids:
            continue
        coords_seq = [item_coords[i] for i in item_ids]
        style = METHOD_STYLES.get(method, {"color": "#000000", "dash": None, "label": method})

        if fmap is None:
            center_lat = sum(c[0] for c in coords_seq) / len(coords_seq)
            center_lon = sum(c[1] for c in coords_seq) / len(coords_seq)
            fmap = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles="cartodbpositron")

        fg = folium.FeatureGroup(name=f"{style['label']} ({len(item_ids)} POIs)", show=True)

        # civic_osrm gets the real routed path; the others get their raw
        # rank-order straight-line path, for a direct visual contrast.
        line_points = osrm_route_geometry(dataset, osrm_profile, coords_seq) if method == "civic_osrm" else coords_seq
        if len(line_points) > 1:
            folium.PolyLine(line_points, color=style["color"], weight=3, opacity=0.85, dash_array=style["dash"]).add_to(fg)

        for rank, (item_id, (lat, lon)) in enumerate(zip(item_ids, coords_seq), start=1):
            is_top1 = rank == 1
            label = f"{style['label']} #{rank}{' ★ top-1' if is_top1 else ''}: {item_id}"
            folium.CircleMarker(
                location=[lat, lon],
                radius=13 if is_top1 else 7,
                color="#000000" if is_top1 else style["color"],
                weight=3 if is_top1 else 1,
                fill=True, fill_color=style["color"],
                fill_opacity=1.0 if is_top1 else 0.9,
                popup=folium.Popup(f"{label}", max_width=220),
                tooltip=label,
            ).add_to(fg)
            if is_top1:
                # small star glyph centered on the marker, on top of everything
                folium.Marker(
                    location=[lat, lon],
                    icon=folium.DivIcon(html=(
                        '<div style="font-size:13px;line-height:26px;width:26px;height:26px;'
                        'text-align:center;color:#ffffff;pointer-events:none;">★</div>'
                    ), icon_size=(26, 26), icon_anchor=(13, 13)),
                ).add_to(fg)

        fg.add_to(fmap)

        hav_total, osrm_total = sequential_distances(item_ids, item_coords, osrm_client=osrm_client)
        stats_rows.append({
            "method": style["label"],
            "n_pois": len(item_ids),
            "top1_item": item_ids[0],
            "sequential_dist_haversine_km": round(hav_total, 2),
            "sequential_dist_osrm_km": round(osrm_total, 2) if osrm_total is not None else None,
        })

    osrm_client.save_cache()

    if fmap is None:
        print(f"No recommendations with known coordinates for user {user_id!r} in {methods}")
        return None, None

    folium.LayerControl(collapsed=False).add_to(fmap)
    stats_df = pd.DataFrame(stats_rows).set_index("method")
    return fmap, stats_df

## Pick a dataset / model / user (hardcoded)

In [ ]:
DATASET = "foursquaretky"  # or "yelpphl"
MODEL_DIRECTORY = sorted(recommender_dirs(DATASET).values())[0]  # or paste an exact directory name
USER_ID = user_ids_for(DATASET, MODEL_DIRECTORY)[0]  # or paste an exact user_id, e.g. "0_x"

print(f"dataset={DATASET!r}  model_directory={MODEL_DIRECTORY!r}  user_id={USER_ID!r}")

fmap, stats_df = build_user_map(DATASET, MODEL_DIRECTORY, USER_ID)
if fmap is not None:
    display(stats_df)
    display(fmap)